# 02 · Dense 基线：Dense-Clean vs. Dense-Aug

| 模型 | 训练方式 | 作用 |
|---|---|---|
| **Dense-Clean** | ImageNet 预训练 ResNet-18，只用干净图片微调 | 参照：没有见过遮挡的模型有多脆弱 |
| **Dense-Aug** | 完全相同的初始权重、batch 顺序、翻转、优化器和 epoch 数；唯一区别是每张训练图以 0.5 概率被遮挡 | **所有剪枝方法共同的起点** |

模型选择：两个模型都选"验证集上各遮挡等级平均准确率最高"的 epoch；测试集从不参与选择。

**如何运行：** 从上到下运行即可（RTX 3070 Ti 上每个模型约 25 秒）。`RETRAIN = False` 时直接读取已保存的结果；改成 `True` 会重新训练并覆盖该 seed 的结果。训练日志会实时显示在单元格下方，同时写入 `results/logs/`。

## 0. 环境检查
任何装好 `requirements.txt` 依赖的 Python 环境都可以（uv / venv / conda 均可，见 README）。
下面这个单元格会把仓库的 `src/` 加入 `sys.path`，所以**不需要**额外安装本项目的包；同时打印当前使用的 Python 路径，请确认它就是你装依赖的那个环境。

In [ ]:
%load_ext autoreload
%autoreload 2
# autoreload：修改 src/occprune 下的代码后无需重启 kernel

import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src" / "occprune").exists() else Path.cwd().parent  # 仓库根目录
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import occprune
print("Python  :", sys.executable)
print("occprune:", occprune.__file__)

## 1. 设置

In [ ]:
import torch
from occprune.utils import load_config
from occprune.pipeline import prepare_data, run_dense, summarize_dense, paired_bootstrap

cfg = load_config()                  # 读取 configs/default.yaml
SEED = cfg["seed"]
RETRAIN = False                      # True = 重新训练两个模型（会覆盖该 seed 的结果）
# cfg["train"]["epochs"] = 25        # 示例：可以在这里修改任意超参数

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
prepare_data(cfg, SEED)              # 若 notebook 01 已运行过，这里会直接复用

## 2. 训练 Dense-Clean（只用干净图片）

In [ ]:
res_clean = run_dense("clean", cfg, SEED, skip_if_exists=not RETRAIN)

## 3. 训练 Dense-Aug（50% 概率随机遮挡）

In [ ]:
res_aug = run_dense("aug", cfg, SEED, skip_if_exists=not RETRAIN)

## 4. 测试集结果
- `Acc`：准确率；`BalAcc`：balanced accuracy（各类召回率平均）；`Drop`：干净准确率减去遮挡准确率（越小越鲁棒，但要和绝对准确率一起看）。
- 遮挡等级的数值是 3 个固定遮挡位置的平均。表格同时保存为 `results/baseline_table_seed42.csv`，图保存在 `figures/`。

In [ ]:
table, figs = summarize_dense(SEED)
cols = ["Method", "Best epoch"] + [c for c in table.columns if c.endswith("Acc") or c.startswith("Drop")]
table[cols].set_index("Method").T

In [ ]:
display(figs["accuracy_vs_occlusion"])   # 左：准确率；右：balanced accuracy
display(figs["training_curves"])         # 训练损失与验证准确率曲线，竖线为选中的 epoch

## 5. 差异是否显著？配对 bootstrap
测试集只有 451 张原图；同一张图的 3 个遮挡版本**不是**独立样本。
因此先把每张图在 3 个遮挡下的结果取平均，再对**原图**做有放回重采样，得到两个模型准确率之差的 95% 置信区间。
置信区间包含 0 说明差异不显著。

In [ ]:
paired_bootstrap("clean", "aug", SEED)   # diff = Dense-Aug − Dense-Clean，单位：百分点

## 6. 与 Taylor 重要性相关的诊断
一阶 Taylor 重要性是 `|w · ∂L/∂w|`。如果模型在**干净**训练图上的 loss 和梯度已经接近 0，那么用干净图片算出的重要性主要是噪声；遮挡图片的 loss 更高、梯度更大，信号更强。
这直接影响 Clean-Taylor、Occlusion-Taylor 和 OAGP 的比较（见 `docs/PROJECT_PLAN.md` 中的 P3）。

- `train_*_loss / acc`：模型在训练集干净图 / 遮挡图上的 loss 和准确率；
- `grad_norm_*`：卷积层权重的梯度范数（BN 处于 eval 模式，batch 大小 16，共 8 个 batch）。

In [ ]:
import pandas as pd
pd.DataFrame({r["method"]: {**r["train_diagnostics"], "best_epoch": r["best_epoch"],
                            "train_time_s": round(r["timing"]["train_time_s"], 1),
                            "device": r["env"].get("gpu") or r["timing"]["device"]}
              for r in (res_clean, res_aug)})